### ***수정된 Prompt***

In [1]:
initial_prompt="""You are a **Scam Video Classifier**. Your task is to analyze the `video_description`, which contains observed screen content, dialogue, captions/OCR, banners, on-screen graphics, actions, and structure. Base all judgments strictly on the provided text; do not guess or hallucinate.

Your objective is to decide whether the video is scam-intended (`is_scam=true`) or normal (`is_scam=false`). Be aware of generalized scam patterns: investment or financial lures, personal data collection or phishing, illegal gambling or trade, religious or psychological inducement, illegal operations or sites, messenger “reading rooms” or external chat funnels, guaranteed or outsized profits, urgent ‘act now/limited time’ calls-to-action, authority or celebrity impersonation, forged ID or license cues, requests for passwords, OTPs, or account/ID numbers, and deposits or wallet transfers.

Normal patterns include information or education, jobs or labor, entertainment or parody, regular ads or branding, and daily life or hobbies **without** coercive or deceptive inducement.

Follow these decision rules:
- If you detect **strong signals** (any clear one), lean towards `is_scam=true`. Strong signals include guaranteed or outsized returns, immediate join/contact/invest calls to action, requests for credentials or payments, external chat or contact funnels (such as Telegram or Kakao), illegal gambling or trading, and celebrity or institution impersonation.
  
- **Moderate signals**—such as charts, “picks,” return discussions, or withdrawal screens—can still be deemed **normal** if they explicitly show an educational or informative context without inducement. 

- If captions or banners serve as warnings or educational elements, treat them as normal **unless** they are paired with simultaneous inducement, personal data requests, payment requests, guaranteed profit claims, or calls to external funnels.

- If information is insufficient or ambiguous, default to `is_scam=false` with low confidence and `"risk":"low"`.

Map the risk level as follows:
- `"high"`: Identifying ≥2 strong signals or direct asks for personal data, money transfers, or external-funnel contacts.
- `"mid"`: Identifying exactly 1 strong signal or a collection of moderate signals that indicate a high likelihood of scams.
- `"low"`: Characterized by weak or ambiguous cues; plausible educational, informational, or branding contexts.

Regarding confidence levels (0.0–1.0):
- Return 0.90–1.00 for multiple consistent strong signals.
- Return 0.70–0.89 for one strong or many moderate signals.
- Return 0.50–0.69 for mixed or weak evidence.
- Return 0.30–0.49 for faint cues; indicate that normal content is more likely.
- Return 0.00–0.29 for no usable cues.

Extract 1–4 **short, verbatim phrases** from the `video_description` as `evidence`, without paraphrasing, ensuring no long spans or duplicates.

When MiniCPM runs later, output your findings in **one strict JSON object only** with the exact keys:
- `"is_scam": true|false`
- `"confidence": number` (0.0–1.0)
- `"risk": "low"|"mid"|"high"`
- `"evidence": ["...", "..."]`
- `"explanation": "..."` (2–4 concise sentences summarizing rationale and risk)

If input is sparse or insufficient, output `"is_scam": false`, set `confidence` to 0.35 or lower, set `"risk":"low"`, and summarize the rationale by mentioning the insufficiency.

Ensure your analysis reflects **safety and consistency**:
- Do not hallucinate brands, people, or numbers not present in the input.
- Normal advertising (product showcases or generic sale language) does not constitute a scam alone; reassess only if accompanied by guaranteed profits, personal data or payment requests, external funnels, or indications of illegality.
- Education, news, or satire regarding scams remains normal unless paired with inducements or data/payment requests."""

### ***테스트 영상 및 라벨***
- 다운로드 받은 모든 테스트 영상에 대한 라벨

In [2]:
import os

abnormal_test_video_dir = '/home/ubuntu/cybercop/video_20251104/abnormal'
normal_test_video_dir = '/home/ubuntu/cybercop/video_20251104/normal'

video_paths = []
truth_labels = []

for root, dirs, files in os.walk(abnormal_test_video_dir):
    for f in files:
        file_path = os.path.join(root, f)
        video_paths.append(file_path)
        truth_labels.append('[[Abnormal]]')

for root, dirs, files in os.walk(normal_test_video_dir):
    for f in files:
        file_path = os.path.join(root, f)
        video_paths.append(file_path)
        truth_labels.append('[[Normal]]')

In [3]:
for video_path, truth_label in zip(video_paths, truth_labels):
    print(video_path, truth_label)

/home/ubuntu/cybercop/video_20251104/abnormal/0017.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0013.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0009.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0000.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0032.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0026.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0008.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0030.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0002.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0037.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0003.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0010.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0012.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0031.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/00

### ***MiniCPM 모델 로드***
- Huggingface 코드스니펫 그대로 적용

In [4]:
import torch
from PIL import Image
from transformers import AutoModel, AutoTokenizer

# load omni model default, the default init_vision/init_audio/init_tts is True
# if load vision-only model, please set init_audio=False and init_tts=False
# if load audio-only model, please set init_vision=False
model = AutoModel.from_pretrained(
    'openbmb/MiniCPM-o-2_6',
    trust_remote_code=True,
    attn_implementation='sdpa', # sdpa or flash_attention_2
    torch_dtype=torch.bfloat16,
    init_vision=True,
    init_audio=True,
    init_tts=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.eval().cuda()
tokenizer = AutoTokenizer.from_pretrained('openbmb/MiniCPM-o-2_6', trust_remote_code=True)

# In addition to vision-only mode, tts processor and vocos also needs to be initialized
model.init_tts()

/home/ubuntu/anaconda3/envs/minicpm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ubuntu/anaconda3/envs/minicpm/lib/python3.10/site-packages/librosa/util/files.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
/home/ubuntu/anaconda3/envs/minicpm/lib/python3.10/site-packages/transformers/models/auto/image_processing_auto.py:513: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(
Loading checkpoint shards: 100%|███████

In [5]:
import math
import numpy as np
from PIL import Image
from moviepy.editor import VideoFileClip
import tempfile
import librosa
import ollama
import re
import traceback
import json
from json import JSONDecodeError

# -------------------------
# Extract video/audio chunks
# -------------------------
def get_video_chunk_content(video_path, flatten=True):
    video = VideoFileClip(video_path)
    print('video_duration:', video.duration)
    
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=True) as temp_audio_file:
        temp_audio_file_path = temp_audio_file.name
        video.audio.write_audiofile(temp_audio_file_path, codec="pcm_s16le", fps=16000)
        audio_np, sr = librosa.load(temp_audio_file_path, sr=16000, mono=True)

    num_units = math.ceil(video.duration)
    contents = []
    for i in range(num_units):
        frame = video.get_frame(i+1)
        image = Image.fromarray((frame).astype(np.uint8))
        audio = audio_np[sr*i:sr*(i+1)]
        if flatten:
            contents.extend(["<unit>", image, audio])
        else:
            contents.append(["<unit>", image, audio])
    return contents

# -------------------------
# MiniCPM inference with confidence
# -------------------------
def run_minicpm(video_path, prompt, model, tokenizer):
    contents = get_video_chunk_content(video_path)
    contents.append("<unit>")
    contents.append(prompt)
    
    sys_msg = model.get_sys_prompt(mode='omni', language='en')
    msg = {"role": "user", "content": contents}
    msgs = [sys_msg, msg]
    
    res = model.chat(
        msgs=msgs,
        tokenizer=tokenizer,
        sampling=True,
        temperature=1.0,
        max_new_tokens=4096,
        omni_input=True,
        use_tts_template=False,
        generate_audio=False,
        max_slice_nums=1,
        use_image_id=False,
        return_dict=True
    )
    
    # Assume MiniCPM can return a confidence score in res["confidence"] (or we can estimate)    
    output_text = res["text"].strip()    
    
    return output_text

# -------------------------
# gpt-oss prompt correction (strong version)
# -------------------------
def correct_prompt_with_llm(old_prompt, additional_prompt):
    instruction = f"""
You are optimizing a prompt for a multimodal LLM to detect scam videos.
MiniCPM received the following prompt but made an incorrect or low-confidence prediction.

Original prompt:
\"\"\"{old_prompt}\"\"\"

{additional_prompt}
"""
    llama_response = ollama.chat(
        model="gpt-oss:20b",
        messages=[{'role': 'user', 'content': instruction}],
        options = {'temperature': 1.0}
    )
    content = llama_response['message']['content']
    return content.strip()

# -------------------------
# Automatic loop with confidence threshold
# -------------------------
def auto_loop(video_path, initial_prompt, truth_label, model, tokenizer, max_iter=5, confidence_thresh=0.9):
    prompt = initial_prompt
    prompts = []
    #for i in range(max_iter):
    output = run_minicpm(video_path, prompt, model, tokenizer)
    # print(f"[MiniCPM] iteration {i+1} output: {output}")
    print(f"[MiniCPM] output: {output}")
    
    try:
        output_dict = json.loads(output)
    except JSONDecodeError:
        additional_prompt = f"There is JSONDecodeError on MiniCPM's output. Revise the prompt so that MiniCPM will correctly classify such videos as {truth_label} scams with high confidence with correct JSON format"
        prompt = correct_prompt_with_llm(prompt, additional_prompt)

    prompts.append(output)

    # pred = "[[Abnormal]]" if output_dict['is_scam'] else "[[Normal]]"
    # conf = output_dict['confidence']
    
    # if pred == truth_label and conf >= confidence_thresh:
    #     print("[OK] Prediction matches truth with high confidence.")
    #     break
    # else:
    #     print("[!] Prediction incorrect or low confidence, correcting prompt...")
    #     additional_prompt = f'Revise the prompt so that MiniCPM will correctly classify such videos as {truth_label} scams with high confidence'
    #     # prompt = correct_prompt_with_llm(prompt, pred, conf, truth_label, additional_prompt)
    #     prompt = initial_prompt
    #     print(f"[!] New optimized prompt:\n{prompt}\n")
    return prompts

import time

final_prompts = []
for video_path, truth_label in zip(reversed(video_paths), reversed(truth_labels)):
# for video_path, truth_label in zip(video_paths, truth_labels):
    data = {'video_path': video_path, 'truth_label': truth_label}
    
    try:        
        print(video_path)
        start = time.perf_counter()
        prompts = auto_loop(
            video_path,
            initial_prompt,
            truth_label,
            model,
            tokenizer,
            max_iter=5,
            confidence_thresh=0.9
        )
        data['prompts'] = prompts    
        duration = time.perf_counter() - start
        data['duration'] = prompts    
        final_prompts.append(data)
    except:
        data['error'] = traceback.format_exc()
        traceback.print_exc()



/home/ubuntu/cybercop/video_20251104/normal/0025.mp4
video_duration: 18.34
MoviePy - Writing audio in /tmp/tmpviytndfj.wav


MoviePy - Done.


The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.


[MiniCPM] output: ```json
{
  "is_scam": true,
  "confidence": 0.95,
  "risk": "high",
  "evidence": [
    "Install and run a scam program, pretending to be an Uber Taxi installer.",
    "Pop-up window showing installation progress with false claims of high profit rates.",
    "Another pop-up warning about the legitimacy of installing this software, indicating it might lead to scams or financial loss."
  ],
  "explanation": "The video shows deceptive tactics commonly used in scam software installations where users are led into believing they will make significant profits from downloading malicious programs."
}
```
/home/ubuntu/cybercop/video_20251104/normal/0011.mp4
video_duration: 34.62
MoviePy - Writing audio in /tmp/tmp7hocsjok.wav


MoviePy - Done.
[MiniCPM] output: {
  "is_scam": false,
  "confidence": 0.55,
  "risk": "low",
  "evidence": [
    "알바하지Street scene scenes",
    "아이폰", 
    "구매", 
    "영수증"
  ],
  "explanation": "The video does not contain any clear or strong signals of scam behavior, such as guaranteed profits, requests for personal data, payment methods, or external chat funnels. It appears to be a normal scenario depicting various everyday tasks and actions without coercive elements."
}
/home/ubuntu/cybercop/video_20251104/normal/0024.mp4
video_duration: 43.1
MoviePy - Writing audio in /tmp/tmpy9ycu3zn.wav


...

MoviePy - Done.
[MiniCPM] output: Based on the description provided, there are strong signals indicating a scam. The video features individuals presenting themselves as doctors or experts in medical and cosmetic procedures while promoting products that promise significant results with minimal effort, such as weight loss within 3 days. These claims align with common scam patterns like guaranteed returns, immediate actions to act now, urgent requests for credentials (like passwords), direct asks for money transfers or wallet deposits, and impersonation of authorities.

Additionally, banners warning against "fake doctor" ads suggest potential scams, reinforcing concerns about deceptive practices.

Given this context:
- `is_scam`: true
- `confidence`: high
- `risk`: mid

`evidence`: 
- Promises of rapid results from minor efforts.
- Presentation by figures claiming expertise but not verified professionally.
- Urgent calls-to-action without educational content.
- Use of branding associated 

MoviePy - Done.


[MiniCPM] output: Based on the provided information, here is a detailed analysis:

### Analysis:
1. **Text and Language**:
   - The text appears to be in Korean.
   - It mentions "2025 MONSTA X CONNECT" which seems like an event or concert.

2. **Seats and Pricing Information**:
   - There are sections indicating different seating areas (e.g., VIP Floor).
   - A specific mention of a discount code ([수원티켓] [MonstaXFan] [돈출 주변 지정석 인증]) for MonstaX Fan, suggesting a special offer related to ticket purchasing.

3. **Scam Indicators**:
   - **High Probability**: Direct references to discounted tickets ("돈출 주변 지정석 인증") suggest a potential scam tactic where victims might be tricked into buying tickets under false pretenses.
   - **Low Confidence**: However, without explicit guarantees or urgent calls-to-action typically found in scams, it leans more towards informational content rather than direct deception.

4. **Educational Context**:
   - Presence of what looks like educational elements ab

MoviePy - Done.


[MiniCPM] output: {
  "is_scam": false,
  "confidence": 0.35,
  "risk": "low",
  "evidence": [
    "therapeutic",
    "therap",
    "therapies"
  ],
  "explanation": "The content appears to be about a therapeutic approach, likely an educational or informational context discussing therapies and related terms. There are no visible indicators of scam activity such as guaranteed profits, requests for credentials or payments, external chat funnels, or indications of illegality."
}
